# 04 - Data Cleaning

## Purpose
This notebook performs data quality checks and cleaning on the combined CRD mortality dataset. Since we used **inner join** in notebook 03, most data quality issues should already be filtered out. This notebook focuses on:

1. Identifying and handling missing values (NaN/empty fields)
2. Detecting and removing Census API error codes (`-666666666`)
3. Engineering percentage features from raw counts
4. Saving the cleaned dataset for feature engineering

## Input
- `data/processed/combined_by_year/combined_all_years.csv`

## Output
- `data/demographics_final/combined_all_years_cleaned_final.csv`

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

## 2. Load Data

In [ ]:
df = pd.read_csv('../data/processed/combined_by_year/combined_all_years.csv')

print(f"Dataset shape: {df.shape}")
print(f"  - Rows (county-year observations): {df.shape[0]:,}")
print(f"  - Columns (features + identifiers): {df.shape[1]}")

## 3. Initial Data Inspection

In [ ]:
print("First 5 rows of the dataset:")
df.head()

In [ ]:
print("Dataset information:")
df.info()

In [ ]:
print(f"Total columns: {len(df.columns)}\n")
print("Column names:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2}. {col}")

## 4. Check for Missing Values

In [ ]:
missing_values = df.isnull().sum()

print("Missing values per column:")
print("=" * 60)
for col in df.columns:
    if missing_values[col] > 0:
        pct = (missing_values[col] / len(df)) * 100
        print(f"{col:40s}: {missing_values[col]:5} ({pct:.2f}%)")

total_missing = missing_values.sum()
print("=" * 60)
print(f"Total missing values: {total_missing}")

if total_missing == 0:
    print("\nNo missing values detected!")

In [ ]:
if df.isnull().any().any():
    rows_with_missing = df[df.isnull().any(axis=1)]
    print(f"Total rows affected: {len(rows_with_missing)}")
    print("\nSample of affected rows:")
    print(rows_with_missing.head(10))
else:
    print("No rows with missing values!")

## 5. Check for Census Error Codes

The Census API uses `-666666666` to indicate data not available.

In [ ]:
error_code_mask = (df == -666666666).any(axis=1)
rows_with_errors = df[error_code_mask]

print("Census API Error Code Check:")
print("=" * 60)
print(f"Rows with -666666666 error codes: {len(rows_with_errors)}")

if len(rows_with_errors) > 0:
    print(f"Percentage of dataset: {(len(rows_with_errors) / len(df)) * 100:.2f}%")
    print("\nColumns affected:")
    for col in df.columns:
        count = (df[col] == -666666666).sum()
        if count > 0:
            print(f"  {col}: {count} occurrences")
else:
    print("No Census error codes found!")

## 6. Remove Problematic Rows

In [ ]:
print("Cleaning dataset...")
print("=" * 60)
print(f"Original dataset: {len(df):,} rows")

df_cleaned = df.dropna()
print(f"After removing missing values: {len(df_cleaned):,} rows ({len(df) - len(df_cleaned)} removed)")

df_cleaned = df_cleaned[(df_cleaned != -666666666).all(axis=1)]
print(f"After removing error codes: {len(df_cleaned):,} rows ({len(df) - len(df_cleaned)} total removed)")

print("=" * 60)
print(f"Final cleaned dataset: {len(df_cleaned):,} rows")
print(f"Rows removed: {len(df) - len(df_cleaned)} ({((len(df) - len(df_cleaned)) / len(df)) * 100:.2f}%)")

## 7. Verify Cleaned Dataset

In [ ]:
print("Cleaned Dataset Verification:")
print("=" * 60)

missing_after = df_cleaned.isnull().sum().sum()
print(f"Missing values: {missing_after}")

error_codes_after = (df_cleaned == -666666666).any().any()
print(f"Error codes remaining: {error_codes_after}")

print(f"\nShape: {df_cleaned.shape}")
print(f"Years present: {sorted(df_cleaned['Year'].unique())}")
print(f"Counties per year:")
print(df_cleaned.groupby('Year').size())

In [ ]:
print("\nSummary Statistics for CRD Mortality Rate:")
print("=" * 60)
print(df_cleaned['crd_mortality_rate'].describe())

## 8. Feature Engineering: Race Percentages

Calculate race percentages from population counts.

In [ ]:
print("Calculating race percentages...")
print("=" * 60)

df_cleaned['White Population (%)'] = (df_cleaned['White Population'] / df_cleaned['Total Population']) * 100
df_cleaned['Hispanic Population (%)'] = (df_cleaned['Hispanic Population'] / df_cleaned['Total Population']) * 100
df_cleaned['Black Population (%)'] = (df_cleaned['Black Population'] / df_cleaned['Total Population']) * 100

print("New columns created:")
print("  - White Population (%)")
print("  - Hispanic Population (%)")
print("  - Black Population (%)")

print("\nSummary statistics:")
print(df_cleaned[['White Population (%)', 'Hispanic Population (%)', 'Black Population (%)']].describe())

## 9. Feature Engineering: Housing and Family Percentages

In [ ]:
print("Calculating housing and family percentages...")
print("=" * 60)

df_cleaned['Households with No Vehicle (%)'] = (
    (df_cleaned['No Vehicle (Owner)'] + df_cleaned['No Vehicle (Renter)']) /
    df_cleaned['Total Occupied Households']
) * 100

df_cleaned['Rent Burden (+50% of HI)'] = (
    df_cleaned['Rent Burden Count (+50%)'] /
    df_cleaned['Rent Denominator']
) * 100

df_cleaned['Single Mother Families (%)'] = (
    df_cleaned['Total Families (Single Mother)'] /
    df_cleaned['Total Families']
) * 100

print("New columns created:")
print("  - Households with No Vehicle (%)")
print("  - Rent Burden (+50% of HI)")
print("  - Single Mother Families (%)")

print("\nSummary statistics:")
print(df_cleaned[['Households with No Vehicle (%)', 'Rent Burden (+50% of HI)', 'Single Mother Families (%)']].describe())

## 10. Drop Raw Count Columns

Remove raw count columns used to create percentages, keeping only Total Population.

In [ ]:
print("Dropping raw count columns...")
print("=" * 60)

columns_to_drop = [
    'White Population',
    'Hispanic Population',
    'Black Population',
    'No Vehicle (Owner)',
    'No Vehicle (Renter)',
    'Total Occupied Households',
    'Rent Burden Count (+50%)',
    'Rent Denominator',
    'Total Families (Single Mother)',
    'Total Families'
]

df_cleaned = df_cleaned.drop(columns=columns_to_drop)

print(f"Dropped {len(columns_to_drop)} columns")
print(f"\nDataset shape after dropping: {df_cleaned.shape}")

## 11. Rename Target Variable

In [ ]:
print("Renaming target variable...")
print("=" * 60)

df_cleaned = df_cleaned.rename(columns={'crd_mortality_rate': 'CRD Mortality Rate'})

print("Renamed 'crd_mortality_rate' to 'CRD Mortality Rate'")
print(f"\nTarget variable summary:")
print(df_cleaned['CRD Mortality Rate'].describe())

## 12. Final Dataset Verification

In [ ]:
print("Final Dataset Structure:")
print("=" * 60)
print(f"Shape: {df_cleaned.shape}")
print(f"  - Rows: {df_cleaned.shape[0]:,}")
print(f"  - Columns: {df_cleaned.shape[1]}")

print("\nAll column names:")
for i, col in enumerate(df_cleaned.columns, 1):
    print(f"{i:2}. {col}")

print("\nSample of final dataset:")
df_cleaned.head(10)

## 13. Save Final Dataset

In [ ]:
output_dir = Path('../data/demographics_final')
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / 'combined_all_years_cleaned_final.csv'
df_cleaned.to_csv(output_path, index=False)

print("=" * 60)
print("DATASET SAVED")
print("=" * 60)
print(f"Final dataset saved to: {output_path}")
print(f"\n  - Shape: {df_cleaned.shape}")
print(f"  - Rows: {df_cleaned.shape[0]:,}")
print(f"  - Columns: {df_cleaned.shape[1]}")
print(f"  - File size: {output_path.stat().st_size / (1024*1024):.2f} MB")